# 🔥 Thermo Cycles — try it in your browser

Simulate **Rankine** (steam and ORC power plants), **refrigeration / heat pump** and **Brayton** (gas turbine)
cycles with real-fluid properties ([CoolProp](http://www.coolprop.org/)), plus an **exergy analysis**.
No installation needed.

**How to use it**
1. Run the *Setup* cell (▶ button on its left). It takes about 20 seconds the first time.
2. In each cycle cell, change the values in the form on the right: the cell runs again automatically
   and shows results, diagrams and where the useful work is lost.

Inputs: pressures in **bar**, temperatures in **°C**, efficiencies in **%**.
Source code, documentation and validation: [github.com/Raffoo26/thermo-cycles](https://github.com/Raffoo26/thermo-cycles)

In [ ]:
#@title ⚙️ Setup (run this first)
!pip install -q CoolProp
import os, sys
if not os.path.exists("thermo-cycles"):
    !git clone -q https://github.com/Raffoo26/thermo-cycles.git
sys.path.insert(0, "thermo-cycles")

import matplotlib.pyplot as plt
from thermocycles import (brayton_cycle, exergy_analysis, rankine_reheat_regen_cycle,
                          refrigeration_cycle)
from thermocycles.plots import plot_exergy, plot_ph, plot_Ts

def show(r, ex):
    r.summary()
    ex.summary()
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
    plot_Ts(r, ax=axes[0]); plot_ph(r, ax=axes[1]); plot_exergy(ex, ax=axes[2])
    fig.tight_layout(); plt.show()

print("Ready ✔")

## Rankine cycle — steam power plant or ORC

In [ ]:
#@title Rankine cycle { run: "auto", display-mode: "form" }
fluid = "Water" #@param ["Water", "R245fa", "R1233zd(E)", "Isopentane", "n-Pentane", "Toluene", "R134a"]
boiler_pressure_bar = 80 #@param {type:"number"}
condenser_pressure_bar = 0.08 #@param {type:"number"}
superheated = True #@param {type:"boolean"}
turbine_inlet_temperature_C = 480 #@param {type:"number"}
reheat = False #@param {type:"boolean"}
reheat_pressure_bar = 7 #@param {type:"number"}
reheat_temperature_C = 440 #@param {type:"number"}
feedwater_heater = False #@param {type:"boolean"}
feedwater_heater_pressure_bar = 7 #@param {type:"number"}
turbine_efficiency_pct = 85 #@param {type:"slider", min:40, max:100, step:1}
pump_efficiency_pct = 85 #@param {type:"slider", min:40, max:100, step:1}
hot_source_temperature_C = 1200 #@param {type:"number"}
ambient_temperature_C = 20 #@param {type:"number"}
net_power_MW = 100 #@param {type:"number"}

try:
    r = rankine_reheat_regen_cycle(
        boiler_pressure_bar, condenser_pressure_bar,
        turbine_inlet_temperature_C if superheated else None,
        reheat_pressure_bar if reheat else None, reheat_temperature_C if reheat else None,
        feedwater_heater_pressure_bar if feedwater_heater else None,
        turbine_efficiency_pct / 100, pump_efficiency_pct / 100, fluid)
    ex = exergy_analysis(r, T0_C=ambient_temperature_C, T_source_C=hot_source_temperature_C)
    show(r, ex)
    if net_power_MW > 0:
        m = r.mass_flow_for_power(net_power_MW * 1e6)
        print(f"\nFor {net_power_MW} MW net: mass flow {m:.1f} kg/s, heat input {m * r.q_in / 1e6:.1f} MW, "
              f"condenser {m * r.q_out / 1e6:.1f} MW")
except ValueError as e:
    print("❌ Inputs not physically possible:", e)

## Refrigeration / heat pump cycle

In [ ]:
#@title Vapour-compression cycle { run: "auto", display-mode: "form" }
refrigerant = "R134a" #@param ["R134a", "R1234yf", "R290", "R600a", "R32", "R410A", "Ammonia"]
evaporating_temperature_C = -10 #@param {type:"number"}
condensing_temperature_C = 40 #@param {type:"number"}
superheat_K = 5 #@param {type:"number"}
subcooling_K = 3 #@param {type:"number"}
compressor_efficiency_pct = 75 #@param {type:"slider", min:40, max:100, step:1}
refrigerated_space_temperature_C = 0 #@param {type:"number"}
ambient_temperature_C = 25 #@param {type:"number"}
cooling_capacity_kW = 3.5 #@param {type:"number"}

try:
    r = refrigeration_cycle(evaporating_temperature_C, condensing_temperature_C, superheat_K,
                            subcooling_K, compressor_efficiency_pct / 100, refrigerant)
    ex = exergy_analysis(r, T0_C=ambient_temperature_C, T_cold_C=refrigerated_space_temperature_C)
    show(r, ex)
    if cooling_capacity_kW > 0:
        m = r.mass_flow_for_cooling(cooling_capacity_kW * 1e3)
        print(f"\nFor {cooling_capacity_kW} kW of cooling: refrigerant {m * 1e3:.2f} g/s, "
              f"compressor {m * r.w_comp / 1e3:.2f} kW, heating as heat pump {m * r.q_cond / 1e3:.2f} kW")
except ValueError as e:
    print("❌ Inputs not physically possible:", e)

## Brayton cycle — gas turbine

In [ ]:
#@title Brayton cycle { run: "auto", display-mode: "form" }
gas = "Air" #@param ["Air", "Nitrogen", "Helium", "CO2"]
pressure_ratio = 10 #@param {type:"number"}
turbine_inlet_temperature_C = 1200 #@param {type:"number"}
compressor_inlet_temperature_C = 15 #@param {type:"number"}
compressor_inlet_pressure_bar = 1.013 #@param {type:"number"}
compressor_efficiency_pct = 86 #@param {type:"slider", min:40, max:100, step:1}
turbine_efficiency_pct = 89 #@param {type:"slider", min:40, max:100, step:1}
regenerator_effectiveness_pct = 0 #@param {type:"slider", min:0, max:100, step:5}
hot_source_temperature_C = 1800 #@param {type:"number"}
ambient_temperature_C = 15 #@param {type:"number"}
net_power_MW = 50 #@param {type:"number"}

try:
    r = brayton_cycle(pressure_ratio, turbine_inlet_temperature_C, compressor_inlet_temperature_C,
                      compressor_inlet_pressure_bar, compressor_efficiency_pct / 100,
                      turbine_efficiency_pct / 100, regenerator_effectiveness_pct / 100, gas)
    ex = exergy_analysis(r, T0_C=ambient_temperature_C, T_source_C=hot_source_temperature_C)
    show(r, ex)
    if net_power_MW > 0:
        m = r.mass_flow_for_power(net_power_MW * 1e6)
        print(f"\nFor {net_power_MW} MW net: gas mass flow {m:.1f} kg/s, heat input {m * r.q_in / 1e6:.1f} MW")
except ValueError as e:
    print("❌ Inputs not physically possible:", e)